<a href="https://colab.research.google.com/github/d86729/os-project-26-2/blob/main/presentation/%EC%A1%B0%EB%B2%88%ED%98%B8_%EC%A3%BC%EC%A0%9C%EB%B2%88%ED%98%B8_%EA%B8%B0%EC%A4%80%EC%84%A0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 통신망 방호 연구: Colab 단일 노트북 시연

이 노트북 하나에 시연에 필요한 파일이 들어 있다. **런타임을 새로 연 뒤 위에서부터 실행**하면 시드 기반 문제 생성, 표준 입력·출력, 공통 점수와 소형 완전탐색을 볼 수 있다. 저장소 복제와 서버 연결은 필요 없다.

**담당:** 김대건 — 문제 생성과 1번 기준선. 신동윤 — 2번 수요 경로 전략. 김연우 — 3번 연결·교환 전략. 진명호 — 4번 공통 평가와 완전탐색.

기준선은 기존 AI 연구 보조 코드다. 아래의 공통 기반 파일은 기존 연구 코드를 담은 것으로, 팀원의 전략 코드와 별도다. 준비 셀은 접혀 있으며 필요하면 펼쳐 소스를 확인할 수 있다.

## 1. 작업 폴더와 공통 기반 파일 준비

Colab에는 일반적으로 NumPy와 SciPy가 설치돼 있다. 없다면 별도 셀에서 `!pip install numpy scipy`로 설치한다. 아래 셀은 `/tmp/network_demo` 폴더를 만들고 거기에 시연 파일을 쓴다. `%%writefile`은 셀 내용을 파일로 저장하는 Colab/Jupyter 문법이다.

### 실행 전 준비

1. 다음 셀에서 실행 환경을 확인한다.
2. 그 다음 셀에서 필요한 패키지만 설치한다. 이미 있으면 설치하지 않는다.
3. 작업 폴더 준비 셀을 실행한 뒤 `%%writefile` 셀들을 위에서부터 실행한다. 파일 생성과 점수 계산에는 앞 단계의 결과가 필요하다. 셀을 건너뛰었다면 해당 준비 단계부터 다시 실행한다.

이미 NumPy가 로드된 환경에서는 스레드 변수를 설정해도 기존 라이브러리에 적용된다고 보장하지 않는다. 이 노트북의 기본 실행은 성능 측정이 아닌 기능 시연이다.

In [1]:
import sys

# Colab 전용 모듈을 불러올 수 있으면 Colab, 없으면 일반 Python 환경입니다.
try:
    import google.colab
    running_in_colab = True
except ImportError:
    running_in_colab = False

print('실행 환경:', 'Google Colab' if running_in_colab else '일반 Python/Jupyter')
print('Python:', sys.version.split()[0])

실행 환경: Google Colab
Python: 3.13.18


In [2]:
import importlib.util
import subprocess
import sys

# import 이름과 설치할 패키지 이름을 짝지어 둡니다.
required = {'numpy': 'numpy', 'scipy': 'scipy'}
# find_spec은 패키지가 있는지 확인합니다. None이면 설치가 필요합니다.
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print('없는 패키지만 설치합니다:', ', '.join(missing))
    # 현재 노트북의 Python으로 pip를 실행합니다. *missing은 목록을 개별 인자로 펼칩니다.
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
else:
    print('필요한 패키지가 이미 설치돼 있어 추가 설치하지 않습니다.')

필요한 패키지가 이미 설치돼 있어 추가 설치하지 않습니다.


In [ ]:
from pathlib import Path
import os
import sys

# 수치 계산 라이브러리가 사용하는 스레드 수를 1로 요청합니다.
# 이미 불러온 라이브러리에는 설정이 적용되지 않을 수 있으므로, 새 런타임에서 먼저 실행하세요.
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[name] = '1'  # 환경 변수 값은 문자열로 지정합니다.

# Colab의 /tmp에 시연 파일을 저장합니다. 런타임이 종료되면 임시 파일이 사라질 수 있습니다.
workdir = Path('/tmp/network_demo')
workdir.mkdir(parents=True, exist_ok=True)  # 이미 폴더가 있어도 오류를 내지 않습니다.
os.chdir(workdir)  # 이후 challenge.txt 같은 상대 경로는 이 폴더를 기준으로 합니다.

# 아래 셀에서 만드는 protection 폴더를 파이썬이 먼저 찾도록 검색 경로에 추가합니다.
if str(workdir) not in sys.path:
    sys.path.insert(0, str(workdir))

# 이전에 불러온 공통 코드를 지워, 다음 실행 때 파일을 다시 읽게 합니다.
for module in list(sys.modules):
    if module == 'protection' or module.startswith('protection.'):
        del sys.modules[module]

Path('protection').mkdir(exist_ok=True)  # 공통 코드 파일을 담을 폴더
print('시연 파일 위치:', workdir)
print('노트북에서 만드는 파일만 이 폴더에 저장합니다.')


아래 공통 기반 파일은 기존 생성·공격·점수 계산을 사용한다. 자료형은 시연에 필요한 정의만 발췌했으며, 공통 처리와 팀원 전략은 분리돼 있다. 접힌 셀을 열면 실제 코드를 볼 수 있다.

In [4]:
%%writefile {workdir}/protection/__init__.py
# 위 %%writefile은 이 셀의 나머지 내용을 지정한 파일에 저장합니다. {workdir}는 작업 폴더입니다.
# 통신망 방호 공통 모듈

Writing /content/network_demo_mwan8xff/protection/__init__.py


In [5]:
%%writefile {workdir}/protection/errors.py
# 위 %%writefile은 이 셀의 나머지 내용을 지정한 파일에 저장합니다. {workdir}는 작업 폴더입니다.
"""Failures are explicit and must never be converted to normal zero scores."""


class InvalidScenarioError(ValueError):
    """Invalid generated or stored research input."""


class InvalidProtectionError(ValueError):
    """Invalid submitted edge identifiers or budget violation."""


class EvaluationError(RuntimeError):
    """Incomplete evaluation; diagnostics are not an official score."""

    def __init__(self, message, *, reason='evaluation_failed', diagnostics=None):
        super().__init__(message)
        self.reason = reason
        self.diagnostics = diagnostics


class DeadlineExceeded(TimeoutError):
    """The caller's monotonic deadline has expired."""

Writing /content/network_demo_mwan8xff/protection/errors.py


In [ ]:
%%writefile {workdir}/protection/models.py
# 위 %%writefile은 이 셀의 나머지 내용을 지정한 파일에 저장합니다. {workdir}는 작업 폴더입니다.
"""시연에 필요한 공통 자료형과 검증 함수만 발췌했다."""
from __future__ import annotations
from dataclasses import dataclass, fields
from decimal import Decimal, ROUND_HALF_EVEN, localcontext
from fractions import Fraction
import math
from protection.errors import EvaluationError, InvalidScenarioError


def require(condition, message):
    if not condition:
        raise InvalidScenarioError(message)


def integer(value, name, minimum=0, maximum=None):
    require(type(value) is int and value >= minimum and
            (maximum is None or value <= maximum), f'{name}: invalid integer {value!r}')


def finite(value, name, minimum=None, maximum=None):
    require(type(value) in (int, float) and math.isfinite(value), f'{name}: finite number required')
    require(minimum is None or value >= minimum, f'{name}: below minimum')
    require(maximum is None or value <= maximum, f'{name}: above maximum')


def fixed_policy(obj):
    for f in fields(obj):
        value = getattr(obj, f.name)
        require(type(value) is type(f.default) and value == f.default,
                f'Unsupported {type(obj).__name__}.{f.name}; version migration required')


@dataclass(frozen=True)
class EvaluationPolicy:
    version: str = 'evaluation-v2'
    rounding: str = 'ROUND_HALF_EVEN'
    decimal_precision: int = 50
    delivery_digits: int = 6
    residual_relative_tolerance: float = 1e-7
    primal_tolerance: float = 1e-9
    dual_tolerance: float = 1e-9
    ipm_tolerance: float = 1e-10
    certificate_denominator_limit: int = 1_000_000_000
    max_solver_attempts: int = 2
    threads: int = 1
    certification: str = 'rational_primal_dual_or_highs_residual_1e-7'

    def __post_init__(self):
        fixed_policy(self)


def coordinate_units(value) -> int:
    with localcontext() as ctx:
        ctx.prec = 50
        value = Decimal(str(value))
        require(value.is_finite(), 'nonfinite coordinate')
        return int(value.quantize(Decimal('0.000000001'), rounding=ROUND_HALF_EVEN) * 10**9)


def rounded_fraction_units(value: Fraction, digits=6) -> int:
    value = Fraction(value) * 10**digits
    whole, remainder = divmod(value.numerator, value.denominator)
    twice = remainder * 2
    return whole + int(twice > value.denominator or (twice == value.denominator and whole % 2 == 1))


def certify_interval(lower: Fraction, upper: Fraction, total: int) -> int:
    if not (total > 0 and 0 <= lower <= upper <= total):
        raise EvaluationError('Invalid primal/dual interval', reason='certificate_failed')
    lo = rounded_fraction_units(lower / total)
    hi = rounded_fraction_units(upper / total)
    if lo != hi:
        raise EvaluationError('Certified interval crosses rounding boundary', reason='numeric_ambiguity')
    return lo


@dataclass(frozen=True)
class GenerationSettings:
    version: str = 'generation-v1'
    alpha: int = 1
    beta: int = 3
    m0: int = 2
    rho: int = 1
    min_distance: float = 1e-9

    def __post_init__(self):
        fixed_policy(self)


@dataclass(frozen=True)
class AttackSettings:
    version: str = 'attack-v1'
    count: int = 1
    radius: float = 0.15
    distance_tolerance: float = 1e-9
    grid_denominator: int = 5
    coordinate_digits: int = 9
    candidate_sources: tuple[str, ...] = ('nodes', 'midpoints', 'grid')

    def __post_init__(self):
        fixed_policy(self)


@dataclass(frozen=True)
class Node:
    id: int
    x: float
    y: float

    def __post_init__(self):
        integer(self.id, 'node id')
        for name in ('x', 'y'):
            v = getattr(self, name)
            finite(v, name, 0)
            require(v < 1, 'coordinates must be <1')
            object.__setattr__(self, name, float(v))


@dataclass(frozen=True)
class Edge:
    id: int
    u: int
    v: int
    capacity_mbps: int
    cost: int

    def __post_init__(self):
        for name in ('id', 'u', 'v'):
            integer(getattr(self, name), name)
        require(self.u < self.v, 'edge requires u<v')
        integer(self.capacity_mbps, 'capacity', 10, 30)
        integer(self.cost, 'cost', 1, 10)


@dataclass(frozen=True)
class Demand:
    id: int
    source: int
    target: int
    amount_mbps: int

    def __post_init__(self):
        for name in ('id', 'source', 'target'):
            integer(getattr(self, name), name)
        require(self.source != self.target, 'demand source==target')
        integer(self.amount_mbps, 'amount', 5, 20)


@dataclass(frozen=True)
class Scenario:
    nodes: tuple[Node, ...]
    edges: tuple[Edge, ...]
    demands: tuple[Demand, ...]
    seed: int
    schema_version: int = 1
    scenario_id: str = ''
    generation: GenerationSettings = GenerationSettings()
    attack_settings: AttackSettings = AttackSettings()
    evaluation_policy: EvaluationPolicy = EvaluationPolicy()
    environment: tuple[tuple[str, str], ...] = ()

    def __post_init__(self):
        integer(self.schema_version, 'schema', 1, 1)
        integer(self.seed, 'seed')
        for name, cls in (('nodes', Node), ('edges', Edge), ('demands', Demand)):
            values = getattr(self, name)
            require(type(values) is tuple and all(type(x) is cls for x in values), name + ': immutable tuple required')
            require(tuple(x.id for x in values) == tuple(range(len(values))), name + ': ids must equal indices')
        n = len(self.nodes)
        require(n >= 3 and len(self.demands) == n, 'n>=3 and exactly n demands required')
        pairs = tuple((e.u, e.v) for e in self.edges)
        require(pairs == tuple(sorted(set(pairs))), 'edges must be unique and sorted')
        require(all(e.v < n for e in self.edges), 'edge endpoint missing')
        demand_pairs = tuple((d.source, d.target) for d in self.demands)
        require(len(set(demand_pairs)) == n, 'duplicate demand pair')
        require(all(max(pair) < n for pair in demand_pairs), 'demand endpoint missing')
        reached = {0}
        while True:
            before = len(reached)
            for u, v in pairs:
                if u in reached or v in reached:
                    reached.update((u, v))
            if len(reached) == before:
                break
        require(len(reached) == n, 'input graph disconnected')
        require(type(self.generation) is GenerationSettings and type(self.attack_settings) is AttackSettings and
                type(self.evaluation_policy) is EvaluationPolicy, 'invalid policy models')
        require(type(self.environment) is tuple and all(type(p) is tuple and len(p) == 2 and
                all(type(x) is str for x in p) for p in self.environment), 'invalid environment')
        require(len({p[0] for p in self.environment}) == len(self.environment), 'duplicate environment key')
        require(type(self.scenario_id) is str, 'invalid scenario_id')
        expected = scenario_hash(self)
        require(not self.scenario_id or self.scenario_id == expected, 'scenario_id mismatch')
        object.__setattr__(self, 'scenario_id', expected)


def environment_tuple(value):
    require(type(value) is tuple and all(type(p) is tuple and len(p) == 2 and
            all(type(x) is str for x in p) for p in value), 'invalid immutable environment')
    require(len({p[0] for p in value}) == len(value), 'duplicate environment key')


def ids_tuple(values):
    require(type(values) is tuple, 'ids require tuple')
    for v in values:
        integer(v, 'edge id')
    require(values == tuple(sorted(set(values))), 'ids must be unique and sorted')


@dataclass(frozen=True)
class SolverDiagnostics:
    success: bool
    status: int
    message: str
    attempts: int
    raw_delivered_mbps: float
    maximum_residual: float
    certified_lower: Fraction | None
    certified_upper: Fraction | None
    certificate_status: str
    elapsed_seconds: float

    def __post_init__(self):
        require(type(self.success) is bool, 'success must be bool')
        require(type(self.message) is str and type(self.certificate_status) is str,
                'diagnostic text must be immutable strings')
        integer(self.status, 'solver status')
        integer(self.attempts, 'attempts', 1, 2)
        for name in ('raw_delivered_mbps', 'maximum_residual', 'elapsed_seconds'):
            finite(getattr(self, name), name, 0)
        if self.certificate_status == 'highs_residual_fallback':
            require(self.success and self.status == 0 and self.maximum_residual <= 1e-7,
                    'fallback requires converged solver and small residual')
            require(self.certified_lower is None and self.certified_upper is None,
                    'fallback is not a rational certificate')
        else:
            require(type(self.certified_lower) is Fraction and type(self.certified_upper) is Fraction,
                    'exact fractions required')
            require(0 <= self.certified_lower <= self.certified_upper, 'invalid certificate interval')


@dataclass(frozen=True)
class Evaluation:
    protected_ids: tuple[int, ...]
    cost: int
    connected_pairs: int
    connected_ratio: float
    delivered_mbps: float
    delivery_ratio: float
    delivery_units: int
    worst_attack: tuple[float, float]

    def __post_init__(self):
        ids_tuple(self.protected_ids)
        integer(self.cost, 'cost')
        integer(self.connected_pairs, 'connected_pairs')
        integer(self.delivery_units, 'delivery_units', 0, 1000000)
        finite(self.connected_ratio, 'connected_ratio', 0, 1)
        finite(self.delivery_ratio, 'delivery_ratio', 0, 1)
        finite(self.delivered_mbps, 'delivered_mbps', 0)
        require(type(self.worst_attack) is tuple and len(self.worst_attack) == 2, 'attack tuple required')
        for coordinate in self.worst_attack:
            finite(coordinate, 'attack coordinate', 0, 1)

    @property
    def score(self):
        return self.connected_pairs, self.delivery_units

    @property
    def defender_key(self):
        return -self.connected_pairs, -self.delivery_units, self.cost, self.protected_ids


@dataclass(frozen=True)
class OptimalReference:
    protected_ids: tuple[int, ...]
    cost: int
    evaluation: Evaluation
    exact_run_id: str
    environment: tuple[tuple[str, str], ...]
    limit_seconds: float
    search_seconds: float
    method: str = 'exhaustive'
    status: str = 'completed'
    optimality_proven: bool = True

    def __post_init__(self):
        ids_tuple(self.protected_ids)
        integer(self.cost, 'cost')
        require(type(self.evaluation) is Evaluation and self.protected_ids == self.evaluation.protected_ids
                and self.cost == self.evaluation.cost, 'reference/evaluation mismatch')
        require(self.method == 'exhaustive' and self.status == 'completed' and self.optimality_proven is True,
                'only completed exhaustive results are references')
        require(type(self.exact_run_id) is str and bool(self.exact_run_id), 'exact_run_id required')
        environment_tuple(self.environment)
        finite(self.limit_seconds, 'limit_seconds', 0)
        finite(self.search_seconds, 'search_seconds', 0)
        require(self.limit_seconds > 0 and self.search_seconds <= self.limit_seconds, 'invalid reference timing')
        require(type(self.environment) is tuple, 'environment tuple required')


@dataclass(frozen=True)
class ScenarioBundle:
    scenario: Scenario
    budget_percent: int
    budget: int
    case_id: str = ''
    optimal_reference: OptimalReference | None = None
    schema_version: int = 1

    def __post_init__(self):
        integer(self.schema_version, 'schema', 1, 1)
        require(type(self.scenario) is Scenario, 'Scenario required')
        integer(self.budget_percent, 'percent')
        require(self.budget_percent in (20, 40), 'percent must be 20 or 40')
        integer(self.budget, 'budget')
        require(self.budget == sum(e.cost for e in self.scenario.edges) * self.budget_percent // 100,
                'budget does not match percent')
        require(self.optimal_reference is None or type(self.optimal_reference) is OptimalReference,
                'invalid reference')
        expected = case_hash(self)
        require(type(self.case_id) is str and (not self.case_id or self.case_id == expected), 'case_id mismatch')
        object.__setattr__(self, 'case_id', expected)
        if self.optimal_reference is not None:
            r = self.optimal_reference
            require(all(i < len(self.scenario.edges) for i in r.protected_ids), 'reference edge missing')
            require(sum(self.scenario.edges[i].cost for i in r.protected_ids) == r.cost <= self.budget,
                    'reference budget/cost invalid')

    def algorithm_input(self):
        return self.scenario, self.budget


def to_data(value):
    from dataclasses import is_dataclass
    if is_dataclass(value):
        return {f.name: to_data(getattr(value, f.name)) for f in fields(value)}
    if isinstance(value, Fraction):
        return {'numerator': value.numerator, 'denominator': value.denominator}
    if type(value) in (tuple, list):
        return [to_data(v) for v in value]
    if type(value) is dict:
        return {k: to_data(v) for k, v in value.items()}
    if type(value) is float:
        require(math.isfinite(value), 'nonfinite JSON')
    require(value is None or type(value) in (str, bool, int, float), 'unsupported JSON value')
    return value


def canonical_json(value):
    import json
    return json.dumps(to_data(value), sort_keys=True, separators=(',', ':'), ensure_ascii=False, allow_nan=False)


def hash_data(value):
    import hashlib
    return hashlib.sha256(canonical_json(value).encode('utf-8')).hexdigest()


def scenario_hash(scenario):
    # 실행 장비 정보보다 재생성 가능한 입력을 식별자로 사용한다.
    return hash_data({name: getattr(scenario, name) for name in
                      ('schema_version', 'generation', 'seed', 'nodes', 'edges', 'demands')})


def case_hash(bundle):
    # 동일 지도라도 예산이나 평가 정책이 다르면 다른 채점 사례다.
    return hash_data({'schema_version': bundle.schema_version, 'scenario_id': bundle.scenario.scenario_id,
                      'attack_settings': bundle.scenario.attack_settings,
                      'evaluation_policy': bundle.scenario.evaluation_policy,
                      'budget_percent': bundle.budget_percent, 'budget': bundle.budget})


def feasible_protection_count(scenario, budget):
    """빈 집합을 포함해 예산 이내의 방호 집합 개수를 독립적으로 센다."""
    counts = [1] + [0] * budget
    for edge in scenario.edges:
        # 비용을 역순으로 순회해야 현재 간선을 같은 집합에 두 번 넣지 않는다.
        for cost in range(budget, edge.cost - 1, -1):
            counts[cost] += counts[cost - edge.cost]
    return sum(counts)

Writing /tmp/network_demo_1a30t9m3/protection/models.py


In [ ]:
%%writefile {workdir}/protection/attacks.py
# 위 %%writefile은 이 셀의 나머지 내용을 지정한 파일에 저장합니다. {workdir}는 작업 폴더입니다.
"""attack-v1의 후보 좌표를 고정하고 후보별 피해 간선을 미리 계산한다."""
from __future__ import annotations
from dataclasses import dataclass
import math
import time
from protection.models import AttackSettings, coordinate_units, integer, finite, ids_tuple, require


@dataclass(frozen=True)
class AttackCandidate:
    id: int
    x: float
    y: float
    damaged_edge_ids: tuple[int, ...]

    def __post_init__(self):
        integer(self.id, 'attack id')
        finite(self.x, 'attack x', 0, 1)
        finite(self.y, 'attack y', 0, 1)
        ids_tuple(self.damaged_edge_ids)


@dataclass(frozen=True)
class PreparedAttacks:
    scenario_id: str
    attack_settings: AttackSettings
    policy_version: str
    candidates: tuple[AttackCandidate, ...]
    preparation_seconds: float

    def __post_init__(self):
        require(type(self.candidates) is tuple and bool(self.candidates), 'attack candidates required')
        require(all(type(c) is AttackCandidate and c.id == i for i, c in enumerate(self.candidates)), 'invalid attack ids')
        coords = tuple((c.x, c.y) for c in self.candidates)
        require(coords == tuple(sorted(set(coords))), 'attack coordinates must be sorted and unique')
        finite(self.preparation_seconds, 'preparation_seconds', 0)


def unique_coordinates(points):
    units = {(coordinate_units(x), coordinate_units(y)) for x, y in points}
    return tuple((x / 10**9, y / 10**9) for x, y in sorted(units))


def segment_distance(point, start, end):
    px, py = point
    ax, ay = start
    dx, dy = end[0] - ax, end[1] - ay
    length_squared = dx * dx + dy * dy
    if length_squared == 0:
        return math.hypot(px - ax, py - ay)
    t = max(0., min(1., ((px - ax) * dx + (py - ay) * dy) / length_squared))
    return math.hypot(px - (ax + t * dx), py - (ay + t * dy))


def is_hit(point, start, end, settings=AttackSettings()):
    return segment_distance(point, start, end) <= settings.radius + settings.distance_tolerance


def prepare_attacks(scenario):
    # 노드·간선 중점·격자점의 좌표를 반올림해 중복을 제거하고 번호를 붙인다.
    started = time.monotonic()
    points = [(n.x, n.y) for n in scenario.nodes]
    for e in scenario.edges:
        u, v = scenario.nodes[e.u], scenario.nodes[e.v]
        points.append(((u.x + v.x) / 2, (u.y + v.y) / 2))
    denominator = scenario.attack_settings.grid_denominator
    points.extend((i / denominator, j / denominator)
                  for i in range(denominator + 1) for j in range(denominator + 1))
    candidates = []
    for i, point in enumerate(unique_coordinates(points)):
        damaged = tuple(e.id for e in scenario.edges if is_hit(
            point, (scenario.nodes[e.u].x, scenario.nodes[e.u].y),
            (scenario.nodes[e.v].x, scenario.nodes[e.v].y), scenario.attack_settings))
        candidates.append(AttackCandidate(i, *point, damaged))
    return PreparedAttacks(scenario.scenario_id, scenario.attack_settings, scenario.evaluation_policy.version,
                           tuple(candidates), time.monotonic() - started)


def surviving_ids(scenario, candidate, protected_ids):
    removed = set(candidate.damaged_edge_ids) - set(protected_ids)
    return tuple(e.id for e in scenario.edges if e.id not in removed)

Writing /tmp/network_demo_1a30t9m3/protection/attacks.py


In [ ]:
%%writefile {workdir}/protection/flow.py
# 위 %%writefile은 이 셀의 나머지 내용을 지정한 파일에 저장합니다. {workdir}는 작업 폴더입니다.
"""방향별 용량과 수요를 희소 선형계획으로 풀고 전달 점수를 인증한다."""
from __future__ import annotations

from dataclasses import dataclass
from fractions import Fraction
import math
from time import monotonic

import numpy as np
from scipy.optimize import linprog
from scipy.sparse import coo_matrix, csr_matrix

from protection.errors import DeadlineExceeded, EvaluationError, InvalidScenarioError
from protection.models import EvaluationPolicy, Scenario, SolverDiagnostics, certify_interval


@dataclass(frozen=True)
class DeliveryResult:
    delivered_mbps: float
    delivery_ratio: float
    delivery_units: int
    diagnostics: SolverDiagnostics


@dataclass(frozen=True)
class LinearProgram:
    """정수 계수로 min c*x를 푼다. G는 용량 제약, A는 흐름 보존 행렬이다.

    처음 demand_count개 변수는 수요별 실제 전달량이다. 목적함수의 해당 계수를
    -1로 두어 합계를 최대화한다. 쌍대 인증을 위해 변수별 상한도 명시한다.
    """
    c: np.ndarray
    G: csr_matrix
    b: np.ndarray
    A: csr_matrix
    upper: np.ndarray
    demand_count: int
    total: int


def _remaining(deadline):
    if deadline is None:
        return None
    remaining = deadline - monotonic()
    if remaining <= 0:
        raise DeadlineExceeded('Delivery evaluation deadline expired')
    return remaining


def _build_lp(scenario, surviving_ids, deadline):
    # 동일 간선의 양방향 용량을 각각 제약하고, 같은 방향의 모든 수요 흐름을 합산한다.
    n, h = len(scenario.nodes), len(scenario.demands)
    arcs = [(u, v, e.capacity_mbps) for i in surviving_ids
            for e in (scenario.edges[i],) for u, v in ((e.u, e.v), (e.v, e.u))]
    m = len(arcs)
    size = h + h*m
    rows, cols, values = [], [], []
    grows, gcols = [], []
    upper = [d.amount_mbps for d in scenario.demands]
    for k, d in enumerate(scenario.demands):
        _remaining(deadline)
        rows.extend((k*n+d.source, k*n+d.target))
        cols.extend((k, k)); values.extend((-1, 1))
        for a, (u, v, capacity) in enumerate(arcs):
            j = h + k*m + a
            rows.extend((k*n+u, k*n+v)); cols.extend((j, j)); values.extend((1, -1))
            grows.append(a); gcols.append(j); upper.append(capacity)
    A = coo_matrix((values, (rows, cols)), shape=(h*n, size), dtype=np.int64).tocsr()
    G = coo_matrix((np.ones(len(grows), dtype=np.int64), (grows, gcols)),
                   shape=(m, size), dtype=np.int64).tocsr()
    return LinearProgram(np.r_[-np.ones(h, dtype=np.int64), np.zeros(h*m, dtype=np.int64)],
                         G, np.array([a[2] for a in arcs], dtype=np.int64), A,
                         np.array(upper, dtype=np.int64), h,
                         sum(d.amount_mbps for d in scenario.demands))


def _failure(message, reason='certificate_failed'):
    raise EvaluationError(message, reason=reason)


def _vectors(lp, result):
    """Check shapes/finite objective and every primal and marginal component."""
    try:
        x, y, z, p, q = (np.asarray(v, dtype=float) for v in
                         (result.x, result.ineqlin.marginals, result.eqlin.marginals,
                          result.lower.marginals, result.upper.marginals))
        expected = (len(lp.c), lp.G.shape[0], lp.A.shape[0], len(lp.c), len(lp.c))
        if any(v.shape != (s,) for v, s in zip((x, y, z, p, q), expected)):
            _failure('Malformed solver vector shape', 'invalid_result')
        if not math.isfinite(float(result.fun)) or any(not np.isfinite(v).all() for v in (x, y, z, p, q)):
            _failure('Nonfinite solver primal/objective/marginal', 'nonfinite_result')
        return x, y, z, p, q
    except (AttributeError, TypeError, ValueError) as exc:
        raise EvaluationError('Missing or malformed solver vectors', reason='invalid_result') from exc


def _check_residuals(lp, result, policy):
    x, *_ = _vectors(lp, result)
    # Absolute residual is recorded; each check uses its own original RHS.
    groups = [(np.abs(lp.A @ x), np.ones(lp.A.shape[0])),
              (np.maximum(lp.G @ x - lp.b, 0), np.maximum(1, np.abs(lp.b))),
              (np.maximum(-x, 0), np.ones(len(x))),
              (np.maximum(x-lp.upper, 0), np.maximum(1, np.abs(lp.upper))),
              (np.array([abs(float(lp.c @ x)-float(result.fun))]),
               np.array([max(1, abs(float(result.fun)))]))]
    maximum = max((float(np.max(r, initial=0)) for r, _ in groups), default=0.)
    if any(np.any(r > policy.residual_relative_tolerance*scale) for r, scale in groups):
        _failure(f'Primal residual exceeds policy tolerance: {maximum}', 'residual_failed')
    return maximum


def _exact_matvec(matrix, vector, deadline):
    matrix = matrix.tocsr()
    result = []
    for row in range(matrix.shape[0]):
        _remaining(deadline)
        result.append(sum((int(matrix.data[j])*vector[matrix.indices[j]]
                           for j in range(matrix.indptr[row], matrix.indptr[row+1])), Fraction(0)))
    return result


def _certify(lp, result, policy, deadline):
    """수치 해를 유리수로 복원하고 원문제와 쌍대의 제약을 정확히 검사한다.

    실행 가능한 원문제 해의 전달량 합은 하한이다. 쌍대 목적값의 부호를
    바꾸면 전달량의 상한이 된다. 두 값 사이에 실제 최적 전달량이 있다.
    """
    def rational_vector(values):
        converted = []
        for value in values:
            _remaining(deadline)
            converted.append(Fraction(str(float(value))).limit_denominator(policy.certificate_denominator_limit))
        return converted
    x, y, z, p, q = (rational_vector(v) for v in _vectors(lp, result))
    if any(v < 0 or v > int(u) for v, u in zip(x, lp.upper)):
        _failure('Exact primal variable bound failed')
    if any(_exact_matvec(lp.A, x, deadline)):
        _failure('Exact primal conservation failed')
    if any(v > int(b) for v, b in zip(_exact_matvec(lp.G, x, deadline), lp.b)):
        _failure('Exact primal capacity failed')
    if any(v > 0 for v in y) or any(v < 0 for v in p) or any(v > 0 for v in q):
        _failure('Exact dual marginal sign failed')
    gy = _exact_matvec(lp.G.T, y, deadline)
    az = _exact_matvec(lp.A.T, z, deadline)
    if any(int(c) != a+b+lo+hi for c, a, b, lo, hi in zip(lp.c, gy, az, p, q)):
        _failure('Exact dual stationarity failed: c != G.T*y + A.T*z + p + q')
    lower = sum(x[:lp.demand_count], Fraction(0))
    upper = -sum((int(b)*v for b, v in zip(lp.b, y)), Fraction(0))
    upper -= sum((int(u)*v for u, v in zip(lp.upper, q)), Fraction(0))
    if not 0 <= lower <= upper <= lp.total:
        _failure('Exact primal/dual interval invalid')
    _remaining(deadline)
    return lower, upper


def maximum_delivery(scenario: Scenario, surviving_ids: tuple[int, ...], *,
                     deadline=None, require_certificate=False) -> DeliveryResult:
    """Return certified delivery on the surviving edge IDs, or an explicit error.

    deadline is an absolute time.monotonic() timestamp, or None (no solver
    time_limit). At most two solves are attempted, first with presolve then
    without. Failed attempts have diagnostic dictionaries, never fake L/U.
    Native hard cancellation still requires the caller's process supervisor.
    """
    start = monotonic()
    if deadline is not None and (type(deadline) not in (int, float) or not math.isfinite(deadline)):
        raise ValueError('deadline must be a finite monotonic timestamp or None')
    _remaining(deadline)
    if (type(surviving_ids) is not tuple or
        any(type(i) is not int or not 0 <= i < len(scenario.edges) for i in surviving_ids) or
        surviving_ids != tuple(sorted(set(surviving_ids)))):
        raise InvalidScenarioError('surviving_ids must be sorted unique valid integer edge IDs')
    lp = _build_lp(scenario, surviving_ids, deadline)
    policy = scenario.evaluation_policy
    history = []
    for attempt in range(1, policy.max_solver_attempts+1):
        options = dict(presolve=attempt == 1, threads=policy.threads,
                       primal_feasibility_tolerance=policy.primal_tolerance,
                       dual_feasibility_tolerance=policy.dual_tolerance,
                       ipm_optimality_tolerance=policy.ipm_tolerance)
        remaining = _remaining(deadline)
        if remaining is not None:
            options['time_limit'] = remaining
        detail = {'attempt': attempt, 'presolve': options['presolve']}
        try:
            try:
                result = linprog(lp.c, A_ub=lp.G, b_ub=lp.b,
                                 A_eq=lp.A, b_eq=np.zeros(lp.A.shape[0]),
                                 bounds=list(zip(np.zeros(len(lp.upper)), lp.upper)),
                                 method='highs', options=options)
            except Exception as exc:
                _remaining(deadline)
                raise EvaluationError(f'Solver raised {type(exc).__name__}: {exc}', reason='solver_failed') from exc
            _remaining(deadline)
            detail.update(status=int(result.status), message=str(result.message))
            if not result.success or result.status != 0:
                _failure(f'LP solver failed: {result.message}', 'solver_failed')
            residual = _check_residuals(lp, result, policy)
            detail.update(raw_delivered_mbps=-float(result.fun), maximum_residual=residual)
            certificate_status = 'certified'
            message = str(result.message)
            try:
                lower, upper = _certify(lp, result, policy, deadline)
                detail.update(certified_lower=lower, certified_upper=upper)
                units = certify_interval(lower, upper, lp.total)
            except EvaluationError as exc:
                if require_certificate or exc.reason not in ('certificate_failed', 'numeric_ambiguity'):
                    raise
                # 일반 제출의 근사 경로도 쌍대 조건과 목적값 잔차가 작을 때만 허용한다.
                x, y, z, p, q = _vectors(lp, result)
                stationarity = lp.G.T @ y + lp.A.T @ z + p + q - lp.c
                dual_objective = float(lp.b @ y + lp.upper @ q)
                residual = max(residual, float(np.max(np.abs(stationarity), initial=0)),
                    float(np.max(y, initial=0)), float(np.max(-p, initial=0)),
                    float(np.max(q, initial=0)), abs(float(result.fun)-dual_objective))
                if residual > 1e-7:
                    raise
                from protection.models import rounded_fraction_units
                units = rounded_fraction_units(Fraction(str(-float(result.fun))) / lp.total)
                lower = upper = None
                certificate_status = 'highs_residual_fallback'
                message += '; rational certificate unavailable: ' + str(exc)
            _remaining(deadline)
            raw = -float(result.fun)
            if raw < 0 or raw > lp.total:
                _failure('Raw delivery outside [0,total]; no clipping allowed')
            diagnostics = SolverDiagnostics(True, int(result.status), message, attempt,
                raw, residual, lower, upper, certificate_status, monotonic()-start)
            _remaining(deadline)
            return DeliveryResult(raw, raw/lp.total, units, diagnostics)
        except EvaluationError as exc:
            detail.update(reason=exc.reason, error=str(exc))
            history.append(detail)
            _remaining(deadline)
            if attempt == policy.max_solver_attempts:
                raise EvaluationError(str(exc), reason=exc.reason,
                                      diagnostics={'attempts': history, 'elapsed_seconds': monotonic()-start}) from exc
    raise AssertionError('Unreachable: fixed policy has two attempts')

Writing /tmp/network_demo_1a30t9m3/protection/flow.py


In [ ]:
%%writefile {workdir}/protection/evaluation.py
# 위 %%writefile은 이 셀의 나머지 내용을 지정한 파일에 저장합니다. {workdir}는 작업 폴더입니다.
"""Strategy-neutral budget, connectivity and worst-attack evaluation."""
import time
from protection.attacks import prepare_attacks, surviving_ids
from protection.flow import maximum_delivery
from protection.errors import DeadlineExceeded
from protection.models import Evaluation
from protection.errors import EvaluationError, InvalidProtectionError
from protection.models import Scenario, integer, require


def budget_for(scenario: Scenario, percent: int) -> int:
    """Return the floored 20% or 40% budget without floating-point arithmetic."""
    integer(percent, 'percent')
    require(percent in (20, 40), 'percent must be 20 or 40')
    return sum(edge.cost for edge in scenario.edges) * percent // 100


def validate_protection(scenario: Scenario, budget: int, edge_ids: list[int]) -> int:
    """Validate a submission without sorting/coercing it; return its total cost."""
    if type(budget) is not int or budget < 0:
        raise InvalidProtectionError('budget must be a nonnegative integer (not bool)')
    if type(edge_ids) is not list:
        raise InvalidProtectionError('protection must be a list[int]')
    previous = -1
    cost = 0
    for edge_id in edge_ids:
        if type(edge_id) is not int or not 0 <= edge_id < len(scenario.edges):
            raise InvalidProtectionError('edge id must be an existing integer id (not bool)')
        if edge_id <= previous:
            raise InvalidProtectionError('edge ids must be strictly increasing and unique')
        previous = edge_id
        cost += scenario.edges[edge_id].cost
    if cost > budget:
        raise InvalidProtectionError('protection cost exceeds budget')
    return cost


def connected_pairs(scenario: Scenario, surviving_ids: tuple[int, ...]) -> int:
    """Count unordered connected node pairs, retaining isolated nodes.

    Internal surviving IDs may be a tuple or list, strictly increasing and
    unique. Invalid internal graph selections raise EvaluationError. Each
    component contributes s*(s-1)//2; the full-graph count n*(n-1)//2 is also
    the denominator to use when displaying the connected ratio.
    """
    if type(surviving_ids) not in (tuple, list):
        raise EvaluationError('surviving ids must be a tuple or list')
    n = len(scenario.nodes)
    total_pairs = n * (n - 1) // 2
    adjacent = [[] for _ in range(n)]
    previous = -1
    for edge_id in surviving_ids:
        if type(edge_id) is not int or not 0 <= edge_id < len(scenario.edges):
            raise EvaluationError('surviving edge id must be an existing integer id (not bool)')
        if edge_id <= previous:
            raise EvaluationError('surviving ids must be strictly increasing and unique')
        previous = edge_id
        edge = scenario.edges[edge_id]
        adjacent[edge.u].append(edge.v)
        adjacent[edge.v].append(edge.u)

    visited = set()
    pairs = 0
    for node in range(n):
        if node in visited:
            continue
        visited.add(node)
        stack = [node]
        size = 0
        while stack:
            current = stack.pop()
            size += 1
            for neighbor in adjacent[current]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    stack.append(neighbor)
        if size == n:
            return total_pairs
        pairs += size * (size - 1) // 2
    return pairs


class Evaluator:
    """One execution's private cache; only fully scored residual networks enter it.

    All connectivity scores are computed before any flow solve. Coordinates
    break attack ties, not traversal order. A new instance starts cold.
    """

    def __init__(self, scenario, prepared=None, *, use_cache=True, require_certificate=False):
        self.scenario = scenario
        self.prepared = prepare_attacks(scenario) if prepared is None else prepared
        if (self.prepared.scenario_id != scenario.scenario_id
                or self.prepared.attack_settings != scenario.attack_settings
                or self.prepared.policy_version != scenario.evaluation_policy.version):
            raise EvaluationError('prepared attacks do not match scenario/policy')
        for candidate in self.prepared.candidates:
            if any(i >= len(scenario.edges) for i in candidate.damaged_edge_ids):
                raise EvaluationError('prepared attack contains unknown edge')
        self.use_cache = use_cache
        self.require_certificate = require_certificate
        self._cache = {}
        self.cache_hits = 0
        self.evaluation_calls = 0
        self.last_diagnostics = ()
        self.fallback_solves = 0
        if require_certificate:
            self._all_edges = (1 << len(scenario.edges)) - 1
            self._attack_masks = {}
            self._connections = {}
            for candidate in self.prepared.candidates:
                mask = sum(1 << edge_id for edge_id in candidate.damaged_edge_ids)
                self._attack_masks.setdefault(mask, candidate)

    @property
    def cache_size(self):
        return len(self._cache)

    @staticmethod
    def _check_deadline(deadline):
        if deadline is not None and time.monotonic() >= deadline:
            raise DeadlineExceeded('evaluation deadline expired')

    def _certified_attack_outcomes(self, edge_ids):
        # 생존 간선이 같으면 점수도 같다. 좌표순 동점 규칙에 따라 첫 공격 위치만 보존한다.
        protected = sum(1 << edge_id for edge_id in edge_ids)
        vulnerable = self._all_edges ^ protected
        survivors = {}
        for damaged, candidate in self._attack_masks.items():
            mask = self._all_edges ^ (damaged & vulnerable)
            survivors.setdefault(mask, candidate)
        for mask, candidate in survivors.items():
            if mask not in self._connections:
                remaining = tuple(edge.id for edge in self.scenario.edges if mask & (1 << edge.id))
                self._connections[mask] = connected_pairs(self.scenario, remaining), remaining
            pairs, remaining = self._connections[mask]
            yield pairs, candidate, remaining

    def evaluate(self, edge_ids, budget, *, deadline=None):
        self._check_deadline(deadline)
        cost = validate_protection(self.scenario, budget, edge_ids)
        self.evaluation_calls += 1
        if self.require_certificate:
            candidates = list(self._certified_attack_outcomes(edge_ids))
        else:
            candidates = []
            for candidate in self.prepared.candidates:
                self._check_deadline(deadline)
                remaining = surviving_ids(self.scenario, candidate, edge_ids)
                candidates.append((connected_pairs(self.scenario, remaining), candidate, remaining))
        minimum = min(pairs for pairs, _, _ in candidates)
        best = None
        diagnostics = []
        for pairs, candidate, remaining in candidates:
            # 연결 쌍이 최소인 공격끼리만 전달량으로 최악의 공격을 가린다.
            if pairs != minimum:
                continue
            self._check_deadline(deadline)
            # 같은 생존 간선 집합은 공격 위치가 달라도 동일한 흐름 문제다.
            key = (self.scenario.scenario_id, sum(1 << i for i in remaining),
                   self.scenario.evaluation_policy.version)
            if self.use_cache and key in self._cache:
                delivery = self._cache[key]
                self.cache_hits += 1
            else:
                if self.require_certificate:
                    delivery = maximum_delivery(self.scenario, remaining, deadline=deadline, require_certificate=True)
                else:
                    delivery = maximum_delivery(self.scenario, remaining, deadline=deadline)
                self._check_deadline(deadline)
                if (delivery.diagnostics is not None and
                        delivery.diagnostics.certificate_status == 'highs_residual_fallback'):
                    self.fallback_solves += 1
                if self.use_cache:
                    self._cache[key] = delivery
            diagnostics.append(delivery.diagnostics)
            attack_key = (delivery.delivery_units, candidate.x, candidate.y)
            if best is None or attack_key < best[0]:
                best = (attack_key, delivery, candidate)
        self._check_deadline(deadline)
        _, delivery, candidate = best
        n = len(self.scenario.nodes)
        self.last_diagnostics = tuple(diagnostics)
        return Evaluation(tuple(edge_ids), cost, minimum, minimum / (n * (n - 1) // 2),
                          delivery.delivered_mbps, delivery.delivery_ratio,
                          delivery.delivery_units, (candidate.x, candidate.y))

Writing /tmp/network_demo_1a30t9m3/protection/evaluation.py


In [ ]:
%%writefile {workdir}/protection/program.py
# 위 %%writefile은 이 셀의 나머지 내용을 지정한 파일에 저장합니다. {workdir}는 작업 폴더입니다.
"""기존 표준 입력 생성과 출력 검증 함수를 그대로 옮긴 시연용 모듈."""
import re
from protection.attacks import prepare_attacks

def scenario_input(bundle):
    """문제 본문의 줄 순서로 입력을 만든다. 최적해와 실행 환경 기록은 전달하지 않는다."""
    scenario = bundle.scenario
    attacks = prepare_attacks(scenario).candidates
    rows = [
        (len(scenario.nodes), len(scenario.edges), len(scenario.demands), len(attacks)),
        (bundle.budget,),
    ]
    rows.extend((node.x, node.y) for node in scenario.nodes)
    rows.extend((e.u, e.v, e.capacity_mbps, e.cost) for e in scenario.edges)
    rows.extend((d.source, d.target, d.amount_mbps) for d in scenario.demands)
    rows.extend((a.x, a.y, len(a.damaged_edge_ids), *a.damaged_edge_ids) for a in attacks)
    return ('\n'.join(' '.join(map(str, row)) for row in rows) + '\n').encode('ascii')


def protected_ids(output, bundle):
    """출력 형식, 간선 번호의 유효성, 오름차순, 방호 비용을 채점 전에 확인한다."""
    from protection.evaluation import validate_protection

    tokens = output.decode('ascii').split()
    if not tokens or len(tokens) > len(bundle.scenario.edges) + 1:
        raise ValueError('출력은 선택 개수와 간선 ID 목록이어야 합니다.')
    if any(re.fullmatch(r'[0-9]{1,10}', token) is None for token in tokens):
        raise ValueError('출력에는 음이 아닌 정수만 사용할 수 있습니다.')
    count, *ids = map(int, tokens)
    if len(ids) != count:
        raise ValueError('선택 개수와 간선 ID 개수가 다릅니다.')
    validate_protection(bundle.scenario, bundle.budget, ids)
    return tuple(sorted(ids))

Writing /tmp/network_demo_1a30t9m3/protection/program.py


## 2. 시드 기반 문제 생성 코드

처음 세 기지를 삼각형으로 잇고, 이후 기지를 기존 기지 두 곳에 연결한다. 기존 기지의 연결 수가 많고 거리가 가까울수록 선택될 가능성이 높다. 비용·용량과 수요도 같은 난수 생성기로 정한다. 다음은 **실제 공통 문제 생성 코드**다.

In [ ]:
%%writefile {workdir}/protection/generation.py
# 위 %%writefile은 이 셀의 나머지 내용을 지정한 파일에 저장합니다. {workdir}는 작업 폴더입니다.
"""generation-v1의 고정 난수 소비 순서로 합성 통신망과 수요를 만든다."""
from __future__ import annotations
import numpy as np
from protection.models import (Node, Edge, Demand, Scenario, GenerationSettings,
                               integer, require)


def connection_probabilities(coordinates, degrees, new_id, candidates):
    distances = np.linalg.norm(coordinates[np.asarray(candidates)] - coordinates[new_id], axis=1)
    weights = np.asarray([degrees[i] for i in candidates], dtype=np.float64) / np.maximum(distances, 1e-9)**3
    total = weights.sum()
    require(np.isfinite(weights).all() and np.isfinite(total) and total > 0,
            'nonfinite or unnormalizable attachment weights')
    probabilities = weights / total
    require(np.isfinite(probabilities).all(), 'invalid attachment probabilities')
    return probabilities


def generate_scenario(n: int, seed: int) -> Scenario:
    integer(n, 'n', 3)
    integer(seed, 'seed')
    rng = np.random.default_rng(seed)
    coordinates = rng.random((n, 2))
    edges = {(0, 1), (0, 2), (1, 2)}
    degrees = [2, 2, 2] + [0] * (n - 3)
    # 삼각형으로 시작해 각 신규 노드를 서로 다른 기존 노드 두 곳에 연결한다.
    for new_id in range(3, n):
        candidates = list(range(new_id))
        for _ in range(2):
            probabilities = connection_probabilities(coordinates, degrees, new_id, candidates)
            selected = int(rng.choice(candidates, p=probabilities))
            edges.add((selected, new_id))
            degrees[selected] += 1
            degrees[new_id] += 1
            candidates.remove(selected)
    ordered = sorted(edges)
    # 정렬한 간선에 용량과 비용을 부여한 후 수요를 추첨한다. 이 순서가 재현성의 일부다.
    capacities = rng.integers(10, 31, size=len(ordered))
    costs = rng.integers(1, 11, size=len(ordered))
    pairs = [(source, target) for source in range(n) for target in range(n) if source != target]
    selection = rng.choice(n * (n - 1), size=n, replace=False)
    amounts = rng.integers(5, 21, size=n)
    return Scenario(
        nodes=tuple(Node(i, float(x), float(y)) for i, (x, y) in enumerate(coordinates)),
        edges=tuple(Edge(i, u, v, int(capacities[i]), int(costs[i])) for i, (u, v) in enumerate(ordered)),
        demands=tuple(Demand(i, *pairs[int(selected)], int(amounts[i])) for i, selected in enumerate(selection)),
        seed=seed, generation=GenerationSettings(),
        environment=(('numpy', np.__version__), ('bit_generator', type(rng.bit_generator).__name__),
                     ('generation_version', 'generation-v1')))

Writing /tmp/network_demo_1a30t9m3/protection/generation.py


아래 예시는 완전탐색으로 정답을 확인한 기지 6개·생성 시드 6·예산 20% 사례다. 생성 시드는 문제 생성에만 쓰고 `challenge.txt`에는 넣지 않는다. 인증된 다른 문제를 쓰려면 해당 문제의 최적 점수도 별도로 확인해야 한다.

In [ ]:
from pathlib import Path
if 'workdir' not in globals() or not (Path(workdir) / 'protection/generation.py').is_file():
    print('먼저 작업 폴더와 공통 파일 생성 셀을 실행하세요.')
else:
    import warnings
    import numpy as np
    from scipy.optimize import OptimizeWarning
    # SciPy가 HiGHS에 스레드 설정을 전달할 때 나오는 안내만 숨깁니다. 다른 오류는 숨기지 않습니다.
    warnings.filterwarnings('ignore', message='Unrecognized options detected:.*', category=OptimizeWarning)
    from protection.generation import generate_scenario
    from protection.evaluation import budget_for
    from protection.models import ScenarioBundle
    from protection.program import scenario_input

    # seed는 문제를 다시 만들기 위한 생성 시드입니다. 전략의 난수를 고정하지 않습니다.
    n, seed, budget_percent = 6, 6, 20
    scenario = generate_scenario(n, seed)
    budget = budget_for(scenario, budget_percent)
    bundle = ScenarioBundle(scenario, budget_percent, budget)
    # 표준 입력에 필요한 내용만 바이트로 변환해 challenge.txt에 씁니다.
    Path('challenge.txt').write_bytes(scenario_input(bundle))
    print(f'기지 {n}개, 선 {len(scenario.edges)}개, 예산 {budget}, 시드 {seed}')
    print('입력 첫 두 줄:')
    print('\n'.join(Path('challenge.txt').read_text().splitlines()[:2]))

기지 6개, 선 9개, 예산 8, 시드 6
입력 첫 두 줄:
6 9 6 51
8


텍스트 입력 형식은 다음과 같다. 번호는 각 구역의 **입력 줄 순서**를 0부터 센다.

1. `기지 수 선 수 수요 수 공격 후보 수`
2. `방호 예산`
3. 기지 수만큼 `x y`
4. 선 수만큼 `기지1 기지2 용량 방호비용`
5. 수요 수만큼 `출발기지 도착기지 요청량`
6. 공격 후보 수만큼 `x y 타격선수 타격선번호...`

출력은 선택한 선의 개수와 오름차순 번호다. 예를 들어 `2 1 4`는 1번과 4번 선을 선택했다는 뜻이다.

## 3. 진명호 · 공통 평가와 완전탐색

완전탐색은 예산 안의 방호안을 모두 생성해 공통 평가기 `Evaluator`로 점수를 계산한다. 제출 답안도 같은 공통 평가기를 사용한다. 아래 프로그램은 작은 문제의 설명용 코드이며, 각 사례의 공식 장시간 인증 기록을 대신하지 않는다.

In [ ]:
%%writefile {workdir}/진명호.py
# 위 %%writefile은 이 셀의 나머지 내용을 지정한 파일에 저장합니다. {workdir}는 작업 폴더입니다.
"""4번 설명용 완전탐색: 공통 평가기로 모든 유효한 방호안을 비교한다.

표준 입력은 팀 전략과 같은 문제 형식이다. 출력은 최적 방호안이다.
작은 사례 시연용이며, 공식 장시간 인증의 실행 기록을 대신하지 않는다.
"""
from itertools import combinations
import sys

from protection.attacks import prepare_attacks
from protection.evaluation import Evaluator
from protection.models import Demand, Edge, Node, Scenario, feasible_protection_count


# 표준 입력을 한 줄씩 읽어 공통 평가기가 사용하는 문제 객체로 바꿉니다.
def read_problem():
    # split()은 공백으로 나누고 map(int, ...)은 각 값을 정수로 바꿉니다.
    n, m, demand_count, attack_count = map(int, sys.stdin.readline().split())
    budget = int(sys.stdin.readline())
    nodes = []
    for node_id in range(n):
        x, y = map(float, sys.stdin.readline().split())
        nodes.append(Node(node_id, x, y))
    edges = []
    for edge_id in range(m):
        start, end, capacity, cost = map(int, sys.stdin.readline().split())
        edges.append(Edge(edge_id, start, end, capacity, cost))
    demands = []
    for demand_id in range(demand_count):
        start, end, amount = map(int, sys.stdin.readline().split())
        demands.append(Demand(demand_id, start, end, amount))
    attacks = []
    for attack_id in range(attack_count):
        fields = sys.stdin.readline().split()
        x, y, count = float(fields[0]), float(fields[1]), int(fields[2])
        damaged = tuple(map(int, fields[3:]))
        if len(damaged) != count:
            raise ValueError('공격의 선 개수가 맞지 않습니다.')
        attacks.append((attack_id, x, y, damaged))
    # 문제는 이미 생성돼 있으므로 이 내부 기본값은 생성에 사용되지 않는다.
    scenario = Scenario(tuple(nodes), tuple(edges), tuple(demands), seed=0)
    prepared = prepare_attacks(scenario)
    expected = [(a.id, a.x, a.y, a.damaged_edge_ids) for a in prepared.candidates]
    if attacks != expected:
        raise ValueError('입력의 공격 목록이 공통 생성 규칙과 다릅니다.')
    return scenario, prepared, budget


def solve(scenario, prepared, budget):
    # 제출 채점과 같은 Evaluator를 사용한다. 참조용은 유리수 인증을 요구한다.
    judge = Evaluator(scenario, prepared, require_certificate=True)
    best = None
    checked = 0
    edge_ids = range(len(scenario.edges))
    # combinations(선 목록, 개수)는 그 개수만큼 서로 다르게 고른 모든 조합입니다.
    for count in range(len(scenario.edges) + 1):
        for choice in combinations(edge_ids, count):
            if sum(scenario.edges[i].cost for i in choice) > budget:
                continue
            result = judge.evaluate(list(choice), budget)
            checked += 1
            # defender_key는 공통 평가기의 점수와 동점 기준을 담은 비교용 값입니다.
            if best is None or result.defender_key < best.defender_key:
                best = result
    if checked != feasible_protection_count(scenario, budget) or judge.fallback_solves:
        raise RuntimeError('모든 방호안의 인증을 완료하지 못했습니다.')
    return best.protected_ids


# 이 파일을 직접 실행할 때만 입력을 읽고 답안을 출력합니다.
if __name__ == '__main__':
    answer = solve(*read_problem())
    # *answer는 목록의 선 번호를 하나씩 펼칩니다: 예) 2 1 4
    print(len(answer), *answer)

Writing /tmp/network_demo_1a30t9m3/진명호.py


In [ ]:
if 'n' not in globals() or not Path('challenge.txt').is_file() or not Path('진명호.py').is_file():
    print('먼저 문제 생성과 진명호.py 파일 생성 셀을 실행하세요.')
elif n <= 6:
    # !는 Colab에서 셸 명령을 실행합니다. <는 입력 파일, >는 출력 파일을 뜻합니다.
    !{sys.executable} -W "ignore:Unrecognized options detected" 진명호.py < challenge.txt > optimal_answer.txt
    print('완전탐색이 고른 선:')
    !cat optimal_answer.txt
else:
    print('시연용 완전탐색은 기지 6개 이하에서만 실행합니다.')

완전탐색이 고른 선:


4 0 5 7 8


## 4. 김대건 · 차수/비용 기준선

양 끝 기지의 연결 수를 합하고, 그 값을 방호 비용으로 나누어 높은 순서로 선을 고른다. 사용한 예산이 제한을 넘지 않게 한다. 다음 `%%writefile` 셀이 실행되면 실제 독립 프로그램 파일이 만들어진다.

In [ ]:
%%writefile {workdir}/기준선.py
# 위 %%writefile은 이 셀의 나머지 내용을 지정한 파일에 저장합니다. {workdir}는 작업 폴더입니다.
"""차수/비용 기준선. AI 보조 초안이며 실제 개인 기여는 별도로 기록한다."""
import sys


def solve():
    # 첫 줄의 개수 네 개를 정수로 읽고, 다음 줄에서 방호 예산을 읽습니다.
    n, m, demand_count, attack_count = map(int, sys.stdin.readline().split())
    budget = int(sys.stdin.readline().split()[0])
    for _ in range(n):
        sys.stdin.readline()  # 기지 좌표: 이 전략에서는 사용하지 않음
    edges = []
    for edge_id in range(m):
        start, end, capacity, cost = map(int, sys.stdin.readline().split())
        edges.append((edge_id, start, end, capacity, cost))
    for _ in range(demand_count + attack_count):
        sys.stdin.readline()  # 수요와 공격: 이 전략에서는 사용하지 않음

    # degree[i]는 i번 기지에 연결된 선의 개수입니다.
    degree = [0] * n
    for edge_id, start, end, capacity, cost in edges:
        degree[start] += 1
        degree[end] += 1

    # 중요도/비용이 같으면 비용과 번호가 작은 선을 먼저 본다.
    # sorted는 키가 작은 선부터 고르므로 중요도 앞에 -를 붙여 높은 선을 먼저 봅니다.
    def priority(edge):
        edge_id, start, end, capacity, cost = edge
        importance = degree[start] + degree[end]
        return (-importance / cost, cost, edge_id)

    selected = []
    for edge_id, start, end, capacity, cost in sorted(edges, key=priority):
        if cost <= budget:
            selected.append(edge_id)
            budget -= cost
    return sorted(selected)


# 직접 실행하면 방호할 선의 개수와 번호를 표준 출력으로 내보냅니다.
if __name__ == '__main__':
    answer = solve()
    print(len(answer), *answer)

Writing /tmp/network_demo_1a30t9m3/기준선.py


In [ ]:
%%writefile {workdir}/매개중심성.py
# 위 %%writefile은 이 셀의 나머지 내용을 지정한 파일에 저장합니다. {workdir}는 작업 폴더입니다.
"""간선 매개중심성/비용 기준선. AI 보조 초안이다."""
# Fraction은 분수를 정확하게 계산해 부동소수점 오차로 동점 순서가 바뀌는 일을 줄입니다.
from fractions import Fraction
import sys


def solve():
    n, m, demand_count, attack_count = map(int, sys.stdin.readline().split())
    budget = int(sys.stdin.readline().split()[0])
    for _ in range(n):
        sys.stdin.readline()
    edges = []
    for edge_id in range(m):
        start, end, capacity, cost = map(int, sys.stdin.readline().split())
        edges.append((edge_id, start, end, capacity, cost))
    for _ in range(demand_count + attack_count):
        sys.stdin.readline()

    # 각 기지에 연결된 (이웃 기지, 선 번호) 목록입니다. _는 사용하지 않는 반복 변수입니다.
    adjacent = [[] for _ in range(n)]
    importance = [Fraction(0) for _ in edges]
    for edge_id, start, end, capacity, cost in edges:
        adjacent[start].append((end, edge_id))
        adjacent[end].append((start, edge_id))

    # 각 기지에서 최단 경로를 찾고, 경로 기여도를 역순으로 더한다.
    for source in range(n):
        previous = [[] for _ in range(n)]
        distance, paths = [-1] * n, [0] * n
        distance[source], paths[source] = 0, 1
        queue = [source]
        # 큐 뒤에 새 기지를 추가하면서 가까운 기지부터 방문합니다(너비 우선 탐색).
        for node in queue:
            for neighbor, edge_id in adjacent[node]:
                if distance[neighbor] == -1:
                    distance[neighbor] = distance[node] + 1
                    queue.append(neighbor)
                if distance[neighbor] == distance[node] + 1:
                    paths[neighbor] += paths[node]
                    previous[neighbor].append((node, edge_id))
        dependency = [Fraction(0) for _ in range(n)]
        # 먼 기지부터 거꾸로 살펴보며 최단 경로의 기여도를 선에 더합니다.
        for node in reversed(queue):
            for parent, edge_id in previous[node]:
                part = Fraction(paths[parent], paths[node]) * (1 + dependency[node])
                importance[edge_id] += part
                dependency[parent] += part

    def priority(edge):
        edge_id, start, end, capacity, cost = edge
        return (-importance[edge_id] / cost, cost, edge_id)

    selected = []
    for edge_id, start, end, capacity, cost in sorted(edges, key=priority):
        if cost <= budget:
            selected.append(edge_id)
            budget -= cost
    return sorted(selected)


if __name__ == '__main__':
    answer = solve()
    print(len(answer), *answer)

Writing /tmp/network_demo_1a30t9m3/매개중심성.py


In [ ]:
from pathlib import Path
if not Path('challenge.txt').is_file() or not Path('기준선.py').is_file():
    print('먼저 문제 생성과 기준선.py 파일 생성 셀을 실행하세요.')
else:
    # !는 Colab에서 셸 명령을 실행합니다. <는 입력 파일, >는 출력 파일을 뜻합니다.
    !python3 기준선.py < challenge.txt > answer.txt
    !cat answer.txt

4 0 5 7 8


`<`는 문제 파일을 표준 입력으로 보내고 `>`는 프로그램의 표준 출력을 답안 파일로 저장한다. Colab 코드 셀에서는 셸 명령 앞에 `!`를 붙인다. Linux 터미널에서는 `!` 없이 실행한다.

팀원이 직접 작성한 프로그램도 같은 형식을 따른다. 예를 들어 별도 셀에 `%%writefile 김연우.py`를 첫 줄로 쓰고 본인 코드를 작성한 뒤, `!python3 김연우.py < challenge.txt > answer.txt`를 실행한다. 아래에는 각 담당 전략의 이해를 돕기 위한 AI 보조 수정본을 넣었다. 담당자가 직접 검토·수정·실행하고 실제 기여를 구분한다.

## 5. 신동윤 · 수요 경로 전략

요청의 출발지와 도착지 사이에 경로를 살리는 데 필요한 선을 함께 살펴본다. 아래는 기존 연구 보조 전략을 이해하기 쉬운 자료구조로 정리한 버전이다.

In [ ]:
%%writefile {workdir}/신동윤.py
# 위 %%writefile은 이 셀의 나머지 내용을 지정한 파일에 저장합니다. {workdir}는 작업 폴더입니다.
"""수요 경로 기반 방호 전략. AI 보조 초안이다."""
import sys

# True이면 한 선씩뿐 아니라 경로를 살리는 선 묶음도 방호 후보로 봅니다.
USE_PATH_BUNDLES = True


def read_problem(stream):
    n, m, d, a = map(int, stream.readline().split())
    budget = int(stream.readline())
    for _ in range(n):
        x, y = map(float, stream.readline().split())
    edges = []
    for _ in range(m):
        u, v, capacity, cost = map(int, stream.readline().split())
        edges.append((u, v, capacity, cost))
    demands = []
    for _ in range(d):
        source, target, amount = map(int, stream.readline().split())
        demands.append((source, target, amount))
    attacks = []
    for _ in range(a):
        tokens = stream.readline().split()
        count = int(tokens[2])
        ids = set(map(int, tokens[3:]))
        assert len(ids) == count
        attacks.append(ids)
    return n, edges, demands, attacks, budget


# 공식 점수를 매번 구하는 대신 연결과 수요 도달량으로 후보를 빠르게 비교합니다.
def surrogate(n, edges, demands, attacks, chosen):
    pairs, reachable = [], []
    for damaged in attacks:
        parent = list(range(n))
        size = [1] * n
        # 같은 연결 요소에 속한 기지들은 같은 대표 번호를 갖습니다.
        def find(x):
            while x != parent[x]:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x
        for eid, (u, v, _, _) in enumerate(edges):
            if eid in damaged and eid not in chosen:
                continue
            ru, rv = find(u), find(v)
            if ru != rv:
                parent[rv] = ru
                size[ru] += size[rv]
        pairs.append(sum(size[i] * (size[i] - 1) // 2 for i in range(n) if find(i) == i))
        reachable.append(sum(amount for s, t, amount in demands if find(s) == find(t)))
    # 공격 수가 같으므로 합계와 평균은 같은 순서가 된다.
    return min(pairs), min(reachable), sum(reachable)


def repair_paths(n, edges, demands, attacks, chosen, remaining):
    """각 공격과 출발 기지에서 추가 방호 비용이 적은 경로를 찾는다."""
    adjacency = [[] for _ in range(n)]
    for eid, (u, v, _, cost) in enumerate(edges):
        adjacency[u].append((v, eid, cost))
        adjacency[v].append((u, eid, cost))
    repairs = []
    sources = sorted({s for s, _, _ in demands})
    targets = {s: sorted({t for u, t, _ in demands if u == s}) for s in sources}
    for damaged in attacks:
        for source in sources:
            # inf는 아직 도달하지 못한 기지에 두는 무한대 초기값입니다.
            best = [(float('inf'), n + 1)] * n
            best[source] = (0, 0)
            previous = [None] * n
            queue = [(0, 0, source)]
            while queue:
                # 추가 비용이 가장 작고, 같으면 거치는 선이 적은 경로부터 살핍니다.
                cost, hops, u = min(queue)
                queue.remove((cost, hops, u))
                if (cost, hops) != best[u]:
                    continue
                for v, eid, price in adjacency[u]:
                    candidate = (cost + (price if eid in damaged and eid not in chosen else 0), hops + 1)
                    if candidate < best[v] and candidate[0] <= remaining:
                        best[v] = candidate
                        previous[v] = (u, eid)
                        queue.append((candidate[0], candidate[1], v))
            for target in targets[source]:
                if best[target][0] > remaining:
                    continue
                extra = set()
                v = target
                while v != source:
                    u, eid = previous[v]
                    if eid in damaged and eid not in chosen:
                        extra.add(eid)
                    v = u
                if extra:
                    repairs.append(tuple(sorted(extra)))

    return repairs


def solve(problem):
    n, edges, demands, attacks, budget = problem
    chosen = set()
    remaining = budget
    current = surrogate(n, edges, demands, attacks, chosen)
    while True:
        # 예산 안에서 추가할 수 있는 한 선과, 설정된 경우 경로 묶음을 후보로 만듭니다.
        candidates = {(eid,) for eid, edge in enumerate(edges)
                      if eid not in chosen and edge[3] <= remaining}
        if USE_PATH_BUNDLES:
            candidates.update(repair_paths(n, edges, demands, attacks, chosen, remaining))
        best_key, best_candidate, best_value = None, None, None
        for addition in sorted(candidates):
            cost = sum(edges[eid][3] for eid in addition)
            value = surrogate(n, edges, demands, attacks, chosen | set(addition))
            if value <= current:
                continue
            # zip은 새 점수와 현재 점수의 같은 위치를 짝지어 비용당 증가량을 계산합니다.
            gain_per_cost = tuple((v - old) / cost for v, old in zip(value, current))
            key = (*gain_per_cost, -cost, tuple(-eid for eid in sorted(addition)))
            if best_key is None or key > best_key:
                best_key, best_candidate, best_value = key, addition, value
        if best_candidate is None:
            break
        chosen.update(best_candidate)
        remaining -= sum(edges[eid][3] for eid in best_candidate)
        current = best_value
    return sorted(chosen)


if __name__ == '__main__':
    answer = solve(read_problem(sys.stdin))
    print(len(answer), *answer)

Writing /tmp/network_demo_1a30t9m3/신동윤.py


## 6. 김연우 · 연결 탐욕과 교환

공격 뒤 기지 연결을 비교해 선을 하나씩 고른다. 그 뒤 선을 추가하거나 바꾸는 작업을 최대 세 번 시도한다. 비트 연산 대신 목록을 사용한다.

In [ ]:
%%writefile {workdir}/김연우.py
# 위 %%writefile은 이 셀의 나머지 내용을 지정한 파일에 저장합니다. {workdir}는 작업 폴더입니다.
"""공격 후 연결을 비교하는 탐욕·교환 전략. AI 보조 초안이다."""
import sys

# 한 번 고른 선을 바꾸거나 추가해 개선하는 과정을 최대 세 번 시도합니다.
IMPROVEMENT_ROUNDS = 3


def solve():
    n, m, demand_count, attack_count = map(int, sys.stdin.readline().split())
    budget = int(sys.stdin.readline().split()[0])
    for _ in range(n):
        sys.stdin.readline()
    edges = []
    for edge_id in range(m):
        start, end, capacity, cost = map(int, sys.stdin.readline().split())
        edges.append((edge_id, start, end, capacity, cost))
    for _ in range(demand_count):
        sys.stdin.readline()
    attacks = []
    for _ in range(attack_count):
        line = sys.stdin.readline().split()
        damaged = tuple(sorted(map(int, line[3:])))
        if damaged not in attacks:
            attacks.append(damaged)

    # 같은 그래프와 방호안을 다시 계산하지 않도록 결과를 딕셔너리에 보관합니다.
    graph_cache = {}
    score_cache = {}

    def graph_value(remaining):
        if remaining in graph_cache:
            return graph_cache[remaining]
        adjacent = [[] for _ in range(n)]
        capacity = 0
        for edge_id in remaining:
            _, start, end, edge_capacity, _ = edges[edge_id]
            adjacent[start].append(end)
            adjacent[end].append(start)
            capacity += edge_capacity
        visited = [False] * n
        pairs = 0
        for start in range(n):
            if visited[start]:
                continue
            visited[start] = True
            # stack에 방문할 기지를 넣고 pop()으로 꺼내 연결된 기지들을 찾습니다.
            stack = [start]
            size = 0
            while stack:
                node = stack.pop()
                size += 1
                for neighbor in adjacent[node]:
                    if not visited[neighbor]:
                        visited[neighbor] = True
                        stack.append(neighbor)
            pairs += size * (size - 1) // 2
        graph_cache[remaining] = (pairs, capacity)
        return pairs, capacity

    def score(selected):
        key = tuple(selected)
        if key not in score_cache:
            results = []
            for damaged in attacks:
                remaining = tuple(i for i in range(m) if i not in damaged or i in selected)
                results.append(graph_value(remaining))
            # 여러 공격의 연결 쌍 수를 정렬한 값과 용량 합으로 후보를 비교합니다. 공식 점수는 아닙니다.
            profile = tuple(sorted(pairs for pairs, capacity in results))
            capacity_sum = sum(capacity for pairs, capacity in results)
            cost = sum(edges[i][4] for i in selected)
            score_cache[key] = (profile, capacity_sum, -cost)
        return score_cache[key]

    def better(candidate, current):
        a, b = score(candidate), score(current)
        return a > b or (a == b and tuple(candidate) < tuple(current))

    selected = []
    # 우선 선을 하나씩 추가하고, 더 나은 선택이 없으면 반복을 멈춥니다.
    while True:
        best = selected
        for edge_id in range(m):
            if edge_id in selected:
                continue
            candidate = sorted(selected + [edge_id])
            if sum(edges[i][4] for i in candidate) <= budget and better(candidate, best):
                best = candidate
        if best == selected:
            break
        selected = best

    # 기존 선 하나를 빼거나 그대로 둔 상태에서 다른 선을 넣어 교환을 시도합니다.
    for _ in range(IMPROVEMENT_ROUNDS):
        best = selected
        for outgoing in [None] + selected:
            reduced = [i for i in selected if i != outgoing]
            for incoming in range(m):
                if incoming in reduced:
                    continue
                candidate = sorted(reduced + [incoming])
                if sum(edges[i][4] for i in candidate) <= budget and better(candidate, best):
                    best = candidate
        if best == selected:
            break
        selected = best
    return selected


if __name__ == '__main__':
    answer = solve()
    print(len(answer), *answer)

Writing /tmp/network_demo_1a30t9m3/김연우.py


In [ ]:
from pathlib import Path
if not all(Path(p).is_file() for p in ('challenge.txt', '신동윤.py', '김연우.py')):
    print('먼저 문제 생성과 각 전략 파일 생성 셀을 실행하세요.')
else:
    # !는 Colab에서 셸 명령을 실행합니다. <는 입력 파일, >는 출력 파일을 뜻합니다.
    !python3 신동윤.py < challenge.txt > answer_2.txt
    !python3 김연우.py < challenge.txt > answer_3.txt
    print('신동윤 전략 출력:')
    !cat answer_2.txt
    print('김연우 전략 출력:')
    !cat answer_3.txt

신동윤 전략 출력:
4 0 5 7 8


김연우 전략 출력:
1 4


## 7. 공통 채점과 최적 참조

선택 번호와 비용을 검사하고 각 공격 후보를 적용한다. 연결된 기지 쌍이 가장 적은 공격, 동점이면 전달률이 가장 낮은 공격의 점수가 방호안의 점수다. 아래의 최적 점수는 이 노트북의 기본 문제에서 완전탐색을 완료한 값이며, 문제 ID가 일치할 때만 비교한다.

In [ ]:
from pathlib import Path
if 'bundle' not in globals() or not Path('answer.txt').is_file():
    print('먼저 문제 생성, 기준선 실행 셀을 실행하세요.')
else:
    from protection.evaluation import Evaluator
    from protection.program import protected_ids

    # 답안의 선 번호와 비용을 검사한 다음 공식 평가기로 채점합니다.
    chosen = protected_ids(Path('answer.txt').read_bytes(), bundle)
    score = Evaluator(scenario).evaluate(list(chosen), budget)
    print('선택한 선:', list(chosen), ' / 사용 비용:', score.cost, '/', budget)
    print('최악 공격 후 연결 쌍:', score.connected_pairs)
    print('최악 공격 후 전달률:', f'{score.delivery_ratio:.2%}')
    # 기존 완전탐색으로 확인한 기본 사례의 ID와 점수다.
    # ID가 같은 문제에만 미리 확정된 최적 점수를 적용합니다.
    certified_id = '1dace74c06e6b9743274a57f82277b95eae51094335ca5532c98c5812dc1b3d1'
    if bundle.case_id == certified_id:
        optimum_score = (10, 571429)
        print('인증 최적 연결 쌍:', optimum_score[0])
        print('인증 최적 전달률:', f'{optimum_score[1] / 1_000_000:.2%}')
        print('최적 점수 일치:', score.score == optimum_score)
    else:
        print('이 문제에 대한 인증 최적 점수가 포함돼 있지 않습니다.')

선택한 선: [0, 5, 7, 8]  / 사용 비용: 7 / 8
최악 공격 후 연결 쌍: 10
최악 공격 후 전달률: 57.14%
인증 최적 연결 쌍: 10
인증 최적 전달률: 57.14%
최적 점수 일치: True


### 담당 전략의 공통 점수 확인

1~3번 전략을 함께 비교하려면 `answer.txt`, `answer_2.txt`, `answer_3.txt`가 모두 필요하다. 파일이 없거나 비어 있으면 채점을 중단하고, 형식이나 예산이 잘못된 답안도 오류로 알린다. 담당자가 코드를 수정하면 해당 전략의 실행 셀부터 다시 실행한다.


In [ ]:
from pathlib import Path
from protection.evaluation import Evaluator
from protection.program import protected_ids

# 세 답안이 모두 있어야 같은 문제에서 세 전략을 비교할 수 있습니다.
answers = [('1번 차수', 'answer.txt'), ('2번 수요 경로', 'answer_2.txt'), ('3번 연결 교환', 'answer_3.txt')]
if 'bundle' not in globals() or 'scenario' not in globals() or 'budget' not in globals():
    raise RuntimeError('먼저 문제 생성 셀을 실행하세요.')

# 누락이나 빈 파일은 채점 전에 모두 확인합니다.
missing = [name for _, name in answers if not Path(name).is_file() or not Path(name).read_bytes().strip()]
if missing:
    raise FileNotFoundError('답안이 없거나 비어 있습니다: ' + ', '.join(missing) + '. 해당 전략을 먼저 실행하세요.')

# 모든 답안의 형식과 예산을 검사한 뒤에만 점수를 출력합니다.
validated = []
for label, filename in answers:
    try:
        ids = protected_ids(Path(filename).read_bytes(), bundle)
    except (ValueError, UnicodeError) as error:
        raise ValueError(f'{label} 답안이 올바르지 않습니다: {error}') from error
    validated.append((label, ids))

judge = Evaluator(scenario)
for label, ids in validated:
    value = judge.evaluate(list(ids), budget)
    print(f'{label}: 선택 {list(ids)}, 연결 {value.connected_pairs}쌍, 전달률 {value.delivery_ratio:.2%}')


## 8. 작은 문제에서 완전탐색 직접 시연

각 선을 선택하거나 선택하지 않는 조합을 만들고 예산 이내인 방호안을 공통 평가기로 계산한다. 연결 쌍 수, 전달 점수, 비용, 선 번호 순의 기존 동점 규칙을 사용한다. 설명과 실행을 위해 **기지 6개 이하에서만** 수행한다. 채점용 공통 구현과 실제 개인 작성 코드는 구분한다.

In [ ]:
# combinations는 지정한 개수의 선을 선택하는 모든 조합을 만듭니다.
from itertools import combinations
from protection.models import feasible_protection_count

if 'scenario' not in globals() or 'budget' not in globals():
    print('먼저 문제 생성 셀을 실행하세요.')
elif n <= 6:
    judge = Evaluator(scenario, require_certificate=True)
    best = None
    evaluated = 0
    for count in range(len(scenario.edges) + 1):
        for picked in combinations(range(len(scenario.edges)), count):
            if sum(scenario.edges[i].cost for i in picked) > budget:
                continue
            value = judge.evaluate(list(picked), budget)
            evaluated += 1
            if best is None or value.defender_key < best.defender_key:
                best = value
    # 누락한 방호안이나 인증되지 않은 계산이 있으면 여기서 오류가 납니다.
    assert evaluated == feasible_protection_count(scenario, budget)
    assert judge.fallback_solves == 0
    print('평가한 방호안:', evaluated)
    print('완전탐색 최적 점수:', best.connected_pairs, '/', f'{best.delivery_ratio:.2%}')
    if 'certified_id' in globals() and bundle.case_id == certified_id:
        print('인증 참조와 최적 점수 일치:', best.score == optimum_score)
else:
    print('시연용 완전탐색은 기지 6개 이하에서만 실행합니다.')

평가한 방호안: 35
완전탐색 최적 점수: 10 / 57.14%
인증 참조와 최적 점수 일치: True


## 면담 준비

1. 같은 시드와 같은 생성 규칙을 쓰는 이유는 무엇인가?
2. 공격 후보는 왜 미리 제공하며, 실제 최악 공격은 어떻게 고르는가?
3. 각자의 전략은 어떤 기준으로 선을 고르고 예산을 지키는가?
4. 연결된 기지 쌍과 전달률은 어떤 순서로 비교하는가?
5. 완전탐색은 무엇을 모두 조사하며, 큰 문제에서는 왜 오래 걸리는가?
6. 최적 참조가 없는 큰 사례에서는 무엇을 비교할 수 있는가?

기본 시연은 시간·메모리 한도를 강제하지 않으며 별도 시험 기록이 아니다. 팀원 전략을 수정한 뒤에는 동일한 사례로 다시 실행해 결과를 기록한다. 이미 NumPy가 불러와진 Colab에서는 스레드 환경 변수가 기존 라이브러리에 적용됐다고 보장하지 않으므로, 시간 측정은 별도로 검증한다.